In [5]:
import pandas as pd
buyers_df=pd.read_json('buyers_cleaned.json')
sales_df=pd.read_csv("sales_cleaned.csv")
agents_df=pd.read_json('agents_cleaned.json')
property_attributes_df=pd.read_json('property_attributes_final_expanded.json')
listings_df=pd.read_json("listings_final_expanded.json")

In [3]:
buyers_df.head()

,buyer_id,sale_id,buyer_type,payment_mode,loan_taken,loan_provider,loan_amount
0,1,L01179,End User,Cash,False,None,0
1,2,L00866,Investor,Cheque,False,None,0
2,3,L00102,Investor,Cheque,True,Axis,2317757
3,4,L00440,Investor,Bank Transfer,False,None,0
4,5,L00059,Investor,UPI,True,HDFC,4191221


In [4]:
sales_df['Date_Sold']=sales_df['Date_Sold'].astype('datetime64[ns]')
sales_df['Sale_Price']=sales_df['Sale_Price'].astype(float).round(2)
sales_df['Listing_ID'].duplicated().sum()
sales_df.head()

,Listing_ID,Sale_Price,Date_Sold,Days_on_Market
0,L01179,925580.00,2023-07-07,65.005560
1,L00866,105416.00,2023-06-14,38.004620
2,L00102,1825184.00,2023-09-09,22.992622
3,L00440,1932085.01,2023-10-29,72.012274
4,L00059,776585.99,2023-05-01,116.000152


In [5]:
agents_df.head()

,Agent_ID,Name,Phone,Email,commission_rate,deals_closed,rating,experience_years,avg_closing_days
0,A0001,Agent A0001,+1-534-665-8373,a0001@realestate.com,2.00,52,4.3,10,64
1,A0002,Agent A0002,+1-493-463-4698,a0002@realestate.com,2.20,26,3.8,20,82
2,A0003,Agent A0003,+1-290-534-1121,a0003@realestate.com,2.85,297,3.6,21,83
3,A0004,Agent A0004,+1-691-610-4878,a0004@realestate.com,1.67,31,4.5,5,61
4,A0005,Agent A0005,+1-829-613-5411,a0005@realestate.com,1.11,198,4.3,16,67


In [6]:
property_attributes_df.head()

,attribute_id,listing_id,bedrooms,bathrooms,floor_number,total_floors,year_built,is_rented,tenant_count,furnishing_status,metro_distance_km,parking_available,power_backup
0,1,L00001,5,3,9,9,2001,True,4,Furnished,7.26,False,True
1,2,L00002,2,2,19,29,2020,False,0,Unfurnished,4.84,True,False
2,3,L00003,2,3,8,26,2007,True,4,Semi-Furnished,4.92,False,False
3,4,L00004,3,3,25,10,2003,False,3,Furnished,1.20,False,True
4,5,L00005,2,2,15,16,2008,True,1,Semi-Furnished,7.90,False,False


In [7]:
listings_df['Price'] = listings_df['Price'].astype(float).round(2)
listings_df.head()

,Listing_ID,City,Property_Type,Price,Sqft,Date_Listed,Agent_ID,Latitude,Longitude
0,L00001,New York,Apartment,1655144.01,2753.009121,2023-05-06,A0015,33.965208,-69.861589
1,L00002,Los Angeles,Apartment,1519141.00,4966.988193,2023-02-14,A0038,42.547892,-90.277860
2,L00003,Houston,Apartment,162489.00,1267.003959,2023-04-22,A0015,28.732327,-115.952982
3,L00004,Phoenix,Apartment,1277015.98,2128.014429,2024-01-02,A0042,26.403938,-74.771490
4,L00005,Phoenix,Townhouse,562297.01,4178.997421,2023-10-29,A0018,39.425252,-83.917878


SQL Lite3

In [6]:
# ==================== COMPLETE CODE: CREATE TABLES & INSERT DATA ====================

import sqlite3

# Connect to database
conn = sqlite3.connect('real_estate.db')
cursor = conn.cursor()

print("=" * 80)
print("CREATING ALL 5 TABLES AND INSERTING DATA")
print("=" * 80)

# ==================== TABLE 1: AGENTS ====================
print("\n1️⃣ Creating AGENTS table...")
cursor.execute('''
CREATE TABLE IF NOT EXISTS agents (
    Agent_ID TEXT PRIMARY KEY,
    Name TEXT,
    Phone TEXT,
    Email TEXT,
    commission_rate REAL,
    deals_closed INTEGER,
    rating REAL,
    experience_years INTEGER,
    avg_closing_days INTEGER
)
''')

# Insert agents data
# DataFrames (e.g., agents_df.to_sql('agents', conn, if_exists='replace', index=False)) to efficiently insert the data from the DataFrames into the newly created tables.
# if_exists='append' means it will add data if the table already has records, and index=False prevents pandas from writing the DataFrame index as a column.
agents_df.to_sql('agents', conn, if_exists='replace', index=False)
cursor.execute("SELECT COUNT(*) FROM agents")
agents_count = cursor.fetchone()[0] #tuple(3,)
print(f"   ✅ AGENTS table created with {agents_count} records")

# ==================== TABLE 2: BUYERS ====================
print("\n2️⃣ Creating BUYERS table...")
cursor.execute('''
CREATE TABLE IF NOT EXISTS buyers (
    buyer_id TEXT PRIMARY KEY,
    sale_id TEXT,
    buyer_type TEXT,
    payment_mode TEXT,
    loan_taken TEXT,
    loan_provider TEXT,
    loan_amount REAL
)
''')

# Insert buyers data
buyers_df.to_sql('buyers', conn, if_exists='replace', index=False)
cursor.execute("SELECT COUNT(*) FROM buyers")
buyers_count = cursor.fetchone()[0]
print(f" ✅ BUYERS table created with {buyers_count} records")

# ==================== TABLE 3: LISTINGS ====================
print("\n3️⃣ Creating LISTINGS table...")
cursor.execute('''
CREATE TABLE IF NOT EXISTS listings (
    Listing_ID TEXT PRIMARY KEY,
    City TEXT,
    Property_Type TEXT,
    Price REAL,
    Sqft REAL,
    Date_Listed TEXT,
    Agent_ID TEXT,
    Latitude REAL,
    Longitude REAL,
    FOREIGN KEY (Agent_ID) REFERENCES agents (Agent_ID)
)
''')

# Insert listings data
listings_df.to_sql('listings', conn, if_exists='replace', index=False)
cursor.execute("SELECT COUNT(*) FROM listings")
listings_count = cursor.fetchone()[0]
print(f"   ✅ LISTINGS table created with {listings_count} records")

# ==================== TABLE 4: SALES ====================
print("\n4️⃣ Creating SALES table...")
cursor.execute('''
CREATE TABLE IF NOT EXISTS sales (
    Listing_ID TEXT,
    Sale_Price REAL,
    Date_Sold TEXT,
    Days_on_Market INTEGER,
    FOREIGN KEY (Listing_ID) REFERENCES listings (Listing_ID)
)
''')

# Insert sales data
sales_df.to_sql('sales', conn, if_exists='replace', index=False)
cursor.execute("SELECT COUNT(*) FROM sales")
sales_count = cursor.fetchone()[0]
print(f"   ✅ SALES table created with {sales_count} records")

# ==================== TABLE 5: PROPERTY_ATTRIBUTES ====================
print("\n5️⃣ Creating PROPERTY_ATTRIBUTES table...")
cursor.execute('''
CREATE TABLE IF NOT EXISTS property_attributes (
    attribute_id TEXT PRIMARY KEY,
    listing_id TEXT,
    bedrooms INTEGER,
    bathrooms INTEGER,
    floor_number INTEGER,
    total_floors INTEGER,
    year_built INTEGER,
    is_rented TEXT,
    tenant_count INTEGER,
    furnishing_status TEXT,
    metro_distance_km REAL,
    parking_available TEXT,
    power_backup TEXT,
    FOREIGN KEY (listing_id) REFERENCES listings (Listing_ID)
)
''')

# Insert property_attributes data
property_attributes_df.to_sql('property_attributes', conn, if_exists='replace', index=False)
cursor.execute("SELECT COUNT(*) FROM property_attributes")
properties_count = cursor.fetchone()[0]
print(f"   ✅ PROPERTY_ATTRIBUTES table created with {properties_count} records")

# ==================== CREATE INDEXES AFTER LOADING DATA ====================
print("\nCreating database indexes for faster queries...")

cursor.execute("""
CREATE INDEX IF NOT EXISTS idx_listings_city_type_price_date
ON listings (City, Property_Type, Price, Date_Listed)
""")

cursor.execute("""
CREATE INDEX IF NOT EXISTS idx_listings_agent_id
ON listings (Agent_ID)
""")

cursor.execute("""
CREATE INDEX IF NOT EXISTS idx_property_attributes_listing_id
ON property_attributes (listing_id)
""")

cursor.execute("""
CREATE INDEX IF NOT EXISTS idx_sales_listing_id
ON sales (Listing_ID)
""")

cursor.execute("""
CREATE INDEX IF NOT EXISTS idx_sales_date_sold
ON sales (Date_Sold)
""")

cursor.execute("""
CREATE INDEX IF NOT EXISTS idx_agents_name
ON agents (Name)
""")

# Commit all changes
conn.commit()
print("✅ Indexes created successfully")



print("\n" + "=" * 80)
print("✅ ALL TABLES CREATED & DATA INSERTED SUCCESSFULLY!")
print("=" * 80)

# ==================== VERIFICATION ====================
print("\n📊 DATABASE SUMMARY:")
print("-" * 80)
print(f"  1. AGENTS              : {agents_count:>8} records")
print(f"  2. BUYERS              : {buyers_count:>8} records")
print(f"  3. LISTINGS            : {listings_count:>8} records")
print(f"  4. SALES               : {sales_count:>8} records")
print(f"  5. PROPERTY_ATTRIBUTES : {properties_count:>8} records")
print(f"  {'─' * 76}")
print(f"  TOTAL                  : {agents_count + buyers_count + listings_count + sales_count + properties_count:>8} records")
print("-" * 80)

# Display sample data from each table
print("\n📋 SAMPLE DATA FROM EACH TABLE:")
print("\n--- AGENTS (First 3 rows) ---")
print(pd.read_sql("SELECT Agent_ID, Name, Phone, commission_rate FROM agents LIMIT 3", conn))

print("\n--- BUYERS (First 3 rows) ---")
print(pd.read_sql("SELECT buyer_id, buyer_type, payment_mode, loan_amount FROM buyers LIMIT 3", conn))

print("\n--- LISTINGS (First 3 rows) ---")
print(pd.read_sql("SELECT Listing_ID, City, Property_Type, Price FROM listings LIMIT 3", conn))

print("\n--- SALES (First 3 rows) ---")
print(pd.read_sql("SELECT Listing_ID, Sale_Price, Date_Sold FROM sales LIMIT 3", conn))

print("\n--- PROPERTY_ATTRIBUTES (First 3 rows) ---")
print(pd.read_sql("SELECT attribute_id, listing_id, bedrooms, bathrooms, year_built FROM property_attributes LIMIT 3", conn))

print("\n" + "=" * 80)

CREATING ALL 5 TABLES AND INSERTING DATA

1️⃣ Creating AGENTS table...
   ✅ AGENTS table created with 50 records

2️⃣ Creating BUYERS table...
 ✅ BUYERS table created with 20000 records

3️⃣ Creating LISTINGS table...
   ✅ LISTINGS table created with 21200 records

4️⃣ Creating SALES table...
   ✅ SALES table created with 720 records

5️⃣ Creating PROPERTY_ATTRIBUTES table...
   ✅ PROPERTY_ATTRIBUTES table created with 21200 records

Creating database indexes for faster queries...
✅ Indexes created successfully

✅ ALL TABLES CREATED & DATA INSERTED SUCCESSFULLY!

📊 DATABASE SUMMARY:
--------------------------------------------------------------------------------
  1. AGENTS              :       50 records
  2. BUYERS              :    20000 records
  3. LISTINGS            :    21200 records
  4. SALES               :      720 records
  5. PROPERTY_ATTRIBUTES :    21200 records
  ────────────────────────────────────────────────────────────────────────────
  TOTAL                  :    